# Install Dependencies

In [ ]:
!pip install -q openai

# Load Modules

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from sklearn.model_selection import train_test_split
import re

client = OpenAI(api_key='API_KEY')

# Define Functions

In [ ]:
def format_prompt(data):

    """

    Background: This function helps format data into a list of dicts into the required shape for fine tuning

    Params:
    data (list): list of dicts

    Returns:
    training_data_list (list): a list in the proper format for converting to jsonl

    """

    training_data_list = []

    for x in data:

        updated_data = {
            "messages": [
                {
                    "role": "system",
                    "content": x['system_message']
                },
                {
                    "role": "user",
                    "content": x['user_content']
                }
            ]
        }

        training_data_list.append(updated_data)

    print(training_data_list)

    return training_data_list

In [ ]:
def convert_to_jsonl_and_save(data_list, filename):

    """
    Background:
    This function converts the data_list provided into a jsonl file

    Params:
    data_list (list): a list of a dict ready to convert to jsonl
    filename (str): the name of the filename we want to convert

    """

    with open(filename, 'w') as file:
        for data_dict in data_list:
            json_str = json.dumps(data_dict)  # Convert dictionary to JSON string
            file.write(json_str + '\n')  # Write to file with a newline

    print(f"✅ Data successfully written to {filename}")

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 20
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("predictions.csv")

# Load Model

In [ ]:
model = 'gpt-4.1-2025-04-14'

# Load Data

In [ ]:
data = pd.read_excel('sampled_sentiment_data.xlsx')

# Zero Shot

## Prepare Prompt

In [ ]:
data['system_message'] = 'You are an AI assistant specialized in climate change sentiment analysis. Classify the sentiment of the following Arabic sentence based on its emotional tone regarding climate change. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.'
data['user_content'] = 'Sentence:\n' + data['Text'] + '\n Predicted Sentiment: '

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-ZeroShot-gpt4-1.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
zero_shot_data = format_prompt(datadict)

In [ ]:
# Save as jsonl
convert_to_jsonl_and_save(zero_shot_data, 'SA-ZeroShot.jsonl')

✅ Data successfully written to SA-ZeroShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= zero_shot_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Neutral'

In [ ]:
true = pd.read_excel('sampled_sentiment_data.xlsx')
y_true = true['sentiment'].values

In [ ]:
zs = pd.DataFrame()

zs['text'] = data2['Text']
zs['Label'] = y_true
zs = zs.reset_index(drop=True)
zs.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(zs, model, zero_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('SA-GPT41-ZeroShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Neutral,368
Negative,122
Positive,11


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
pred['Prediction'].shape[0]

501

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      print(answer)
      preds.append('None')

In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(zs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.493

f1_score:  0.4274875821192206
\precision:  0.6305954232272835
ecall:  0.4930139720558882

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.3940    0.8683    0.5421       167
    Negative     0.7705    0.5629    0.6505       167
    Positive     0.7273    0.0479    0.0899       167

    accuracy                         0.4930       501
   macro avg     0.6306    0.4930    0.4275       501
weighted avg     0.6306    0.4930    0.4275       501



# Few Shot

In [ ]:
data = pd.read_excel('sampled_sentiment_data.xlsx')

## Prepare Prompt

In [ ]:
data['system_message'] = '''I want to divide this into system message and user contern
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Sentiment: Neutral'''
data['user_content'] = 'The sentence you need to classify\nSentence:\n' + data['Text'] +'\nPredicted Sentiment:'

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-FewShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
few_shot_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(few_shot_data, 'SA-FewShot.jsonl')

✅ Data successfully written to SA-FewShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= few_shot_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'Sentiment: Negative'

In [ ]:
y_true = data2['sentiment'].values

In [ ]:
fs = pd.DataFrame()

fs['text'] = data2['Text']
fs['Label'] = y_true
fs = fs.reset_index(drop=True)
fs.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(fs, model, few_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('SA-gpt41-FewShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
Sentiment: Negative,238
Sentiment: Positive,160
Sentiment: Neutral,98
System message:\nYou are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following,4
Negative,1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      print(answer)
      preds.append('None')

System message:
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following
System message:
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following
System message:
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following
System message:
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following


In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'None', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral', 'None']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2, 'None':3}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(fs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive', 'None'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.723

f1_score:  0.7117554570390728
\precision:  0.7467914780975152
ecall:  0.7225548902195609

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.7959    0.4671    0.5887       167
    Negative     0.6695    0.9581    0.7882       167
    Positive     0.7750    0.7425    0.7584       167
        None     0.0000    0.0000    0.0000         0

    accuracy                         0.7226       501
   macro avg     0.5601    0.5419    0.5338       501
weighted avg     0.7468    0.7226    0.7118       501



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.p

# CoT

## Prepare Prompt

In [ ]:
data['system_message'] = '''You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Step 1: Read the sentence
Carefully read the sentence to fully understand its meaning, context, and tone. Consider both explicit statements and any implied emotional cues.

Step 2: Identify Emotionally Charged Language
- Highlight positive language, such as words indicating satisfaction, happiness, or praise (e.g., great, amazing, love, well-done).
- Highlight negative language, such as words indicating dissatisfaction, frustration, or criticism (e.g., terrible, hate, broken, disappointing).
- Neutral statements neither praise nor criticize.

Step 3: Analyze the Emotional Balance
Consider the overall tone and intent of the sentence, including sarcasm or contrast.

Step 4: Determine Sentiment
- If positive sentiment dominates, classify as Positive.
- If negative sentiment dominates, classify as Negative.
- If there is no clear emotional direction or the content is purely factual, classify as Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Thoughts:
- Positive words: فرحة, اجمل, الخير
- Negative words: None
- Emotional Balance: tweet uses only positive language and promotes an uplifting message about generosity and the return of happiness.
- Sentiment: positive

Predicted Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت


Thoughts:
- Positive word: None
- Negative words: سئمت, دمعة
- Emotional Balance: tweet uses only negative language which revolves around fatigue, sadness, and being overwhelmed by memories.
- Sentiment: Negative

Predicted Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Thoughts:
- Positive words: جميلاً
- Negative words: ألماً
- Emotional Balance: tweet mentions both positive and negative outcomes, the overall tone is cautionary and moralistic, not emotionally expressive.
- Sentiment: Neutral
'''
data['user_content'] = 'The sentence you need to classify\nSentence:\n' + data['Text']

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('SA-CoT-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
CoT_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(CoT_data, 'SA-CoT.jsonl')

✅ Data successfully written to SA-CoT.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= CoT_data[7]['messages'],
      temperature = 0.2,
      max_tokens= 512
  )

response.choices[0].message.content

'Thoughts:\n- Positive words: None\n- Negative words: موقف صعب, محتاج مساعده (indicate difficulty and need for help)\n- Emotional Balance: The sentence expresses distress and a request for assistance, which are negative emotional cues.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative'

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("SA-CoT-predictions-temp0.2.csv")

In [ ]:
y_true = data2['sentiment'].values

In [ ]:
cot = pd.DataFrame()

cot['text'] = data2['Text']
cot['Label'] = y_true
cot = cot.reset_index(drop=True)
cot.head()

,text,Label
0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative
1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative
2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative
3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative
4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative


In [ ]:
store_predictions(cot, model, CoT_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('SA-CoT-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
"Thoughts:\n- Positive words: له ثقله, سيولد قوه, كبير. كبير. كبير, 💚\n- Negative words: النقص\n- Emotional Balance: While the sentence acknowledges a negative aspect (النقص في الاهلي), it quickly reframes it as a source of strength (سيولد قوه) and emphasizes the important, positive role of the fans (دورنا كبير). The repetition of ""كبير"" and the heart emoji further reinforce a positive, supportive tone.\n\nPredicted Sentiment: Positive",1
"Thoughts:\n- Positive words: None\n- Negative words: ضلم, قمعه, قتله, العصابة, انتقم الله منه\n- Emotional Balance: The sentence is filled with strong negative language, expressing anger, accusation, and a wish for retribution. There is clear dissatisfaction and condemnation towards Macron and the mentioned parties.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: وش ذنبه يعيش هالحياة ؟ (What is his fault to live this life?), 💔 (broken heart emoji), نهاية الحب (end of love)\n- Emotional Balance: The sentence expresses sadness, blame, and disappointment about the end of love, using a broken heart emoji and questioning the fairness of someone's suffering.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: امراض, هم البناء, سكر, ضغط, اكبر من مجرد تفكير (all indicate stress, illness, and negative consequences)\n- Emotional Balance: The sentence focuses on the negative impact and stress associated with building a house, mentioning diseases and pressure.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: unfair, 😫 (frustration), 😭😭 (crying emojis indicating disappointment or sadness)\n- Emotional Balance: The sentence expresses frustration and disappointment about finishing exercise only to find a neighbor bringing delicious food, which is described as ""unfair."" The use of negative emojis reinforces the negative emotional tone.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
...,...
"Thoughts:\n- Positive words: رضاكم هو هدفنا (your satisfaction is our goal) – this is a standard customer service phrase, not an expression of actual satisfaction.\n- Negative words: ماوصلت شحنتي (my shipment has not arrived yet), وللان (until now) – indicates dissatisfaction and frustration.\n- Emotional Balance: The sentence expresses frustration about a delayed shipment despite payment. The phrase ""رضاكم هو هدفنا"" is used ironically, highlighting the gap between the company's slogan and the customer's experience.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: الاراضي المسروقة (stolen lands), خط احمر (red line, implying untouchable or protected in a critical way)\n- Emotional Balance: The sentence questions whether the fenced lands of princes are considered among the stolen lands, then sarcastically suggests they are a ""red line"" (untouchable). The tone is critical and implies dissatisfaction with perceived injustice or privilege.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1
"Thoughts:\n- Positive words: None\n- Negative words: موقف صعب, محتاج مساعده (indicate difficulty and need for help)\n- Emotional Balance: The sentence expresses that the speaker is in a difficult situation and needs assistance, which reflects distress and vulnerability.\n- Sentiment: Negative\n\nPredicted Sentiment: Negative",1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
def get_text_after_sentiment(row):
    text = row['Prediction']
    word = 'Predicted Sentiment:'
    idx = text.find(word)
    if idx != -1:
        return text[idx + len(word):].strip()
    else:
      print(text)
      return 'Neutral'

# Apply to create new column
pred['Normalized Output'] = pred.apply(get_text_after_sentiment, axis=1)

pred.head()

Thoughts:
- Positive words: فديتك, الله يعوضني و يعوضك, واشوف بصحة و عافيه (expressions of affection, hope, and well-wishing)
- Negative words: خانني الوقت, طارت الطيارة (expressions of disappointment and missed opportunity)
- Emotional Balance: The sentence expresses disappointment about missing a flight after a week of preparation, but also includes affectionate language and hopeful prayers for compensation and health. The negative emotion of disappointment is dominant, but it is softened by positive wishes.

Sentiment: Negative


,Unnamed: 0,text,Label,Prediction,Normalized Output
0,0,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
1,1,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
2,2,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
3,3,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative
4,4,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...,Negative,Thoughts:\n- Positive words: None\n- Negative ...,Negative


In [ ]:
pred['Normalized Output'].value_counts()

,count
Normalized Output,
Negative,227
Positive,165
Neutral,109


In [ ]:
pred['Normalized Output'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Normalized Output']:
    if (
          "Positive" in answer
          or "positive" in answer
          or "pos" in answer
          or "ايجابي" in answer
          or "ايجابية" in answer
      ):
        preds.append("Positive")
    elif (
          "Negative" in answer
          or "negative" in answer
          or "neg" in answer
          or "سلبي" in answer
          or "سلبية" in answer
      ):
        preds.append("Negative")
    elif (
          "Neutral" in answer
          or "neutral" in answer
          or "neu" in answer
          or "حيادي" in answer
          or "حيادية" in answer
      ):
        preds.append("Neutral")
    else:
      preds.append('None')

In [ ]:
np.unique(preds)

array(['Negative', 'Neutral', 'Positive'], dtype='<U8')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['Positive', 'Negative', 'Neutral']
mapping = {'Neutral': 0, 'Negative': 1, 'Positive':2}
def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(cot.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=['Neutral', 'Negative', 'Positive'])
print('\nClassification Report:')
print(class_report)

Accuracy: 0.747

f1_score:  0.7365245577321424
\precision:  0.7586728539078698
ecall:  0.7465069860279441

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.7982    0.5210    0.6304       167
    Negative     0.6960    0.9461    0.8020       167
    Positive     0.7818    0.7725    0.7771       167

    accuracy                         0.7465       501
   macro avg     0.7587    0.7465    0.7365       501
weighted avg     0.7587    0.7465    0.7365       501

